In [3]:

import json, re, os
from collections import Counter, defaultdict

JSONL_PATH = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/bechcicki_products_all.jsonl"

# Załaduj JSONL
bech = []
with open(JSONL_PATH) as f:
    for line in f:
        try:
            bech.append(json.loads(line.strip()))
        except:
            pass

print(f"Produkty bechcicki.pl: {len(bech)}")
print(f"Pola: {list(bech[0].keys()) if bech else 'brak'}")
print()

# Jakie kategorie mamy?
cats = Counter(p.get('category_main','') for p in bech)
print("TOP 15 kategorii w JSONL:")
for cat, n in cats.most_common(15):
    print(f"  {cat}: {n}")

print()
# Przykład danych
print("Przykład produktu:")
p = bech[0]
for k,v in p.items():
    print(f"  {k}: {v!r}")


Produkty bechcicki.pl: 16480
Pola: ['sku', 'name', 'brand', 'manufacturer_index', 'ean', 'url', 'category_main', 'category_sub', 'category_leaf', 'alt_id', '_page']

TOP 15 kategorii w JSONL:
  farby-i-rozpuszczalniki: 4846
  chemia-budowlana: 3436
  izolacje: 2673
  narzedzia-i-mocowania: 1769
  stropy-i-sciany: 1287
  dachy: 974
  sucha-zabudowa: 847
  plytki: 648

Przykład produktu:
  sku: 'P-0029063'
  name: ''
  brand: 'Henkel'
  manufacturer_index: '1687904'
  ean: '5900089112241'
  url: ''
  category_main: 'chemia-budowlana'
  category_sub: 'kleje'
  category_leaf: 'kleje-do-glazury'
  alt_id: '0033978 KLEJ DO GRESU C2TE CM12 PLUS 25KG CERESIT 1687904 HENKEL'
  _page: 1


In [7]:

import asyncio, aiohttp, json, re

SANITY_PROJECT = "nzcwegq7"
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
DATASET = "production"

async def sanity_query(query: str, params: dict = {}):
    url = f"https://{SANITY_PROJECT}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
    params_str = json.dumps(params)
    async with aiohttp.ClientSession() as session:
        async with session.get(
            url,
            params={"query": query, "$": params_str} if params else {"query": query},
            headers={"Authorization": f"Bearer {SANITY_TOKEN}"}
        ) as resp:
            data = await resp.json()
            return data.get("result", [])

# Pobierz próbkę produktów z Sanity żeby sprawdzić format SKU
sample_q = '*[_type == "product"][0...20]{_id, name, sku, "catSlug": category->slug.current}'
sample = await sanity_query(sample_q)
print(f"Próbka Sanity ({len(sample)} produktów):")
for p in sample[:5]:
    print(f"  _id={p['_id'][:20]}, sku={p.get('sku')!r}, name={p.get('name','')[:50]!r}")


Próbka Sanity (20 produktów):
  _id=prod-grunt-ceresit-c, sku='CRS-CT16-10L', name='Grunt głęboko penetrujący Ceresit CT 16 10L'
  _id=prod-hydroizolacja-c, sku='CRS-CL50-15', name='Elastyczna hydroizolacja Ceresit CL 50 15kg'
  _id=prod-klej-styropian-, sku='WEB-THERM-EX-25', name='Zaprawa klejąca weber.therm extra 25kg'
  _id=prod-styropian-swiss, sku='SWP-EPS70-031-15', name='Styropian grafitowy Swisspor EPS 70-031 Fasada 15c'
  _id=prod-tynk-gipsowy-kn, sku='KNF-GB-25', name='Tynk gipsowy Knauf Goldband 25kg'


In [11]:

# Sprawdź czy JSONL ma URLs + sprawdź EAN coverage
with_url = [p for p in bech if p.get('url')]
with_ean = [p for p in bech if p.get('ean')]
with_name = [p for p in bech if p.get('name')]

print(f"Produkty z URL: {len(with_url)}")
print(f"Produkty z EAN: {len(with_ean)}")
print(f"Produkty z nazwą: {len(with_name)}")

# Przykłady z URL
if with_url:
    print("\nPrzykłady URL:")
    for p in with_url[:3]:
        print(f"  {p['url']}")
        
# Przykłady z EAN + nazwą
farby = [p for p in bech if p.get('category_main') == 'farby-i-rozpuszczalniki' and p.get('ean') and p.get('name')]
print(f"\nFarby z EAN+nazwą: {len(farby)}")
if farby:
    print("Przykład:")
    p = farby[0]
    for k,v in p.items():
        print(f"  {k}: {v!r}")


Produkty z URL: 0
Produkty z EAN: 14965
Produkty z nazwą: 0

Farby z EAN+nazwą: 0


In [15]:

import re, json
from collections import defaultdict

# alt_id: "0033978 KLEJ DO GRESU C2TE CM12 PLUS 25KG CERESIT 1687904 HENKEL"
# Format: [alt_id] [NAZWA PRODUKTU] [manufacturer_index] [BRAND]

def parse_alt_id(alt_id: str, brand: str, manufacturer_index: str) -> dict:
    if not alt_id:
        return {}
    # Usuń manufacturer_index i brand ze środka
    text = alt_id.strip()
    
    # Wyciągnij jednostkę z nazwy (25KG, 10L, 5L, 15CM itp.)
    unit_match = re.search(r'\b(\d+(?:[,\.]\d+)?)\s*(KG|L|LT|ML|M2|M3|SZT|OPK|PAK|CM|MM|G)\b', text, re.I)
    unit = ''
    if unit_match:
        unit = f"{unit_match.group(1)}{unit_match.group(2).upper()}"
    
    # Pełna nazwa produktu — usuń alt_id numer z początku
    clean = re.sub(r'^\d{5,8}\s+', '', text)
    # Usuń trailing manufacturer_index i brand
    if manufacturer_index:
        clean = clean.replace(manufacturer_index, '').strip()
    if brand:
        clean = re.sub(re.escape(brand.upper()) + r'\s*$', '', clean, flags=re.I).strip()
    
    # Capitalize properly
    name_clean = re.sub(r'\s+', ' ', clean).strip().title()
    
    return {'name_full': name_clean, 'unit': unit, 'unit_raw': unit_match.group(0) if unit_match else ''}

# Testuj
test_cases = [
    ("0033978 KLEJ DO GRESU C2TE CM12 PLUS 25KG CERESIT 1687904 HENKEL", "Henkel", "1687904"),
    ("0063420 FARBA ELEWACYJNA AKRYLOWA SNIEZKA ELEWATOR 5L 1452300 SNIEZKA", "Sniezka", "1452300"),
    ("0005012 STYROPIAN GRAFITOWY EPS 70-031 FASADA 15CM SWISSPOR 8007002 SWISSPOR", "Swisspor", "8007002"),
]
for alt_id, brand, mfr in test_cases:
    r = parse_alt_id(alt_id, brand, mfr)
    print(f"Input: {alt_id[:60]}")
    print(f"  name: {r.get('name_full')}")
    print(f"  unit: {r.get('unit')}")
    print()

# Zbuduj lookup: EAN → {name, brand, unit, category, url}
# bechcicki.pl URL format: /produkt/[slug] — spróbuj zrekonstruować
ean_lookup = {}
for p in bech:
    alt = p.get('alt_id', '')
    ean = p.get('ean', '')
    brand = p.get('brand', '')
    mfr = p.get('manufacturer_index', '')
    cat_main = p.get('category_main', '')
    cat_sub = p.get('category_sub', '')
    cat_leaf = p.get('category_leaf', '')
    
    parsed = parse_alt_id(alt, brand, mfr)
    
    if ean:
        ean_lookup[ean] = {
            'ean': ean,
            'sku': p.get('sku', ''),
            'brand': brand,
            'name_full': parsed.get('name_full', ''),
            'unit': parsed.get('unit', ''),
            'cat_main': cat_main,
            'cat_sub': cat_sub,
            'cat_leaf': cat_leaf,
            'manufacturer_index': mfr,
        }

print(f"\nLookup EAN→dane: {len(ean_lookup)} wpisów")
# Zapisz lookup do pliku
with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/ean_lookup.json', 'w') as f:
    json.dump(ean_lookup, f, ensure_ascii=False, indent=2)
print("Zapisano: ean_lookup.json")

# Sprawdź farby
farby_ean = {k:v for k,v in ean_lookup.items() if v['cat_main'] == 'farby-i-rozpuszczalniki'}
print(f"\nFarby w lookupie: {len(farby_ean)}")
for k, v in list(farby_ean.items())[:3]:
    print(f"  EAN={k}: name={v['name_full']!r}, unit={v['unit']!r}, brand={v['brand']!r}")


Input: 0033978 KLEJ DO GRESU C2TE CM12 PLUS 25KG CERESIT 1687904 HE
  name: Klej Do Gresu C2Te Cm12 Plus 25Kg Ceresit
  unit: 25KG

Input: 0063420 FARBA ELEWACYJNA AKRYLOWA SNIEZKA ELEWATOR 5L 145230
  name: Farba Elewacyjna Akrylowa Sniezka Elewator 5L
  unit: 5L

Input: 0005012 STYROPIAN GRAFITOWY EPS 70-031 FASADA 15CM SWISSPOR 
  name: Styropian Grafitowy Eps 70-031 Fasada 15Cm Swisspor
  unit: 15CM




Lookup EAN→dane: 14256 wpisów
Zapisano: ean_lookup.json

Farby w lookupie: 4245
  EAN=5903973231684: name='Ftal.Biała Podk. 0,8L Urekor D Podkładowa', unit='0,8L', brand='Ftalowa'
  EAN=5903973116653: name='Ftal.Biała Podk. 1L Supermal Xurekor Dx Podkładowa Ftalowa Tt25', unit='1L', brand='&&'
  EAN=5904000058601: name="Emul. Classic White 5L It'S", unit='5L', brand='Simple'


In [19]:

import asyncio, aiohttp, json, re, sys, time
sys.path.insert(0, "/app/skills/common/scripts")
from batch_llm import batch_llm

SANITY_PROJECT = "nzcwegq7"
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
DATASET = "production"

SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"

# ── Helpery ──────────────────────────────────────────────────────────────────

def extract_unit_from_name(name: str) -> str:
    """Wyciąga jednostkę miary z nazwy produktu (25kg, 10L, 5L, 15cm itp.)"""
    pattern = r'\b(\d+(?:[,\.]\d+)?)\s*(kg|KG|l|L|lt|LT|ml|ML|m2|M2|m3|M3|szt|SZT|g\b|G\b|cm|CM|mm|MM|op|OP)\b'
    m = re.search(pattern, name)
    if m:
        num = m.group(1).replace(',', '.')
        unit = m.group(2).lower()
        # Normalizuj: L→L, kg→kg, ml→ml
        unit_map = {'l': 'L', 'lt': 'L', 'kg': 'kg', 'ml': 'ml', 
                    'm2': 'm²', 'm3': 'm³', 'szt': 'szt', 'cm': 'cm',
                    'mm': 'mm', 'g': 'g', 'op': 'op'}
        return f"{num}{unit_map.get(unit, unit)}"
    return ''

def extract_tags_from_name_and_category(name: str, cat_slug: str) -> list:
    """Generuje tagi na podstawie nazwy i kategorii"""
    tags = []
    name_lower = name.lower()
    cat_lower = (cat_slug or '').lower()
    
    # Farby
    if 'farba' in name_lower or 'farby' in cat_lower or 'farba' in cat_lower:
        if 'elewac' in name_lower:
            tags.append('farba-elewacyjna')
        elif 'akryl' in name_lower:
            tags.append('farba-akrylowa')
        elif 'lateks' in name_lower:
            tags.append('farba-lateksowa')
        elif 'ceramicz' in name_lower:
            tags.append('farba-ceramiczna')
        elif 'beton' in name_lower:
            tags.append('farba-do-betonu')
        elif 'metal' in name_lower or 'antykoroz' in name_lower:
            tags.append('farba-do-metalu')
        elif 'drewno' in name_lower or 'drewna' in name_lower:
            tags.append('farba-do-drewna')
        elif 'podłog' in name_lower or 'posadzk' in name_lower:
            tags.append('farba-do-podlogi')
        elif 'emulsj' in name_lower:
            tags.append('farba-emulsyjna')
        else:
            tags.append('farba-wewnetrzna')
    
    # Kleje
    if 'klej' in name_lower or 'kleje' in cat_lower:
        if 'styropian' in name_lower or 'etics' in name_lower or 'therm' in name_lower:
            tags.append('klej-do-styropianu')
        elif 'gres' in name_lower or 'glazur' in name_lower or 'plytk' in name_lower:
            tags.append('klej-do-plytek')
        elif 'montaz' in name_lower:
            tags.append('klej-montazowy')
        elif 'tapeta' in name_lower:
            tags.append('klej-tapetowy')
        else:
            tags.append('klej-budowlany')
    
    # Tynki
    if 'tynk' in name_lower:
        if 'gips' in name_lower:
            tags.append('tynk-gipsowy')
        elif 'maszyno' in name_lower:
            tags.append('tynk-maszynowy')
        elif 'silikat' in name_lower:
            tags.append('tynk-silikatowy')
        elif 'silikonow' in name_lower:
            tags.append('tynk-silikonowy')
        elif 'akryl' in name_lower:
            tags.append('tynk-akrylowy')
        else:
            tags.append('tynk-mineralny')
    
    # Izolacje
    if 'styropian' in name_lower:
        if 'grafitow' in name_lower:
            tags.append('styropian-grafitowy')
        elif 'fasad' in name_lower:
            tags.append('styropian-fasadowy')
        elif 'podlog' in name_lower:
            tags.append('styropian-podlogowy')
        else:
            tags.append('styropian')
    
    if 'wełna' in name_lower or 'welna' in name_lower:
        tags.append('welna-mineralna')
    
    # Grunty
    if 'grunt' in name_lower or 'gruntow' in name_lower:
        tags.append('grunt-budowlany')
    
    # Uszczelniacze / hydroizolacja
    if 'uszczelni' in name_lower or 'hydroizol' in name_lower:
        tags.append('hydroizolacja')
    
    # Zaprawy
    if 'zaprawa' in name_lower or 'zapraw' in name_lower:
        if 'murar' in name_lower:
            tags.append('zaprawa-murarska')
        elif 'wyrów' in name_lower or 'poziom' in name_lower:
            tags.append('zaprawa-wyrownujaca')
        else:
            tags.append('zaprawa-budowlana')
    
    return list(set(tags))

async def sanity_query_all(query: str) -> list:
    url = f"https://{SANITY_PROJECT}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
    async with aiohttp.ClientSession() as session:
        async with session.get(url, params={"query": query},
                               headers={"Authorization": f"Bearer {SANITY_TOKEN}"}) as resp:
            data = await resp.json()
            return data.get("result", [])

async def sanity_patch(doc_id: str, patch_data: dict, session: aiohttp.ClientSession) -> bool:
    url = f"https://{SANITY_PROJECT}.api.sanity.io/v2021-10-21/data/mutate/{DATASET}"
    mutation = {
        "mutations": [{
            "patch": {
                "id": doc_id,
                "set": patch_data,
            }
        }]
    }
    async with session.post(url, json=mutation,
                             headers={"Authorization": f"Bearer {SANITY_TOKEN}",
                                      "Content-Type": "application/json"}) as resp:
        return resp.status == 200

# ── Krok 1: Pobierz produkty z Sanity ────────────────────────────────────────
print("Pobieranie produktów z Sanity...")
products_q = '*[_type == "product"][0...3000]{_id, name, sku, unit, shortDescription, description, tags, "catSlug": category->slug.current, "catName": category->name}'
products = await sanity_query_all(products_q)
print(f"Pobrano: {len(products)} produktów")

# Filtruj te, które wymagają wzbogacenia
need_enrichment = [p for p in products if not p.get('shortDescription') and p.get('name')]
need_unit = [p for p in products if not p.get('unit') and p.get('name')]
print(f"Bez shortDescription: {len(need_enrichment)}")
print(f"Bez unit: {len(need_unit)}")

# ── Krok 2: Wyciągnij jednostki z nazw ───────────────────────────────────────
print("\nEkstrakcja jednostek z nazw produktów...")
unit_updates = {}
tag_updates = {}

for p in products:
    name = p.get('name', '')
    pid = p['_id']
    cat_slug = p.get('catSlug', '')
    
    # Unit
    if not p.get('unit') or len(p.get('unit', '')) > 10:
        extracted_unit = extract_unit_from_name(name)
        if extracted_unit:
            unit_updates[pid] = extracted_unit
    
    # Tags
    if not p.get('tags') or len(p.get('tags', [])) == 0:
        extracted_tags = extract_tags_from_name_and_category(name, cat_slug)
        if extracted_tags:
            tag_updates[pid] = extracted_tags

print(f"Produkty z wyciągniętą jednostką: {len(unit_updates)}")
print(f"Produkty z wygenerowanymi tagami: {len(tag_updates)}")

# ── Krok 3: Wygeneruj krótkie opisy dla top 500 produktów ────────────────────
print("\nGenerowanie krótkich opisów (batch_llm)...")
# Priorytyzuj produkty z popularnych kategorii
priority_cats = ['farby-rozpuszczalniki', 'farby-i-rozpuszczalniki', 'chemia-budowlana', 
                 'izolacje', 'stropy-sciany', 'sucha-zabudowa']
need_desc = [p for p in need_enrichment if p.get('catSlug') in priority_cats][:500]
print(f"Do generowania opisów: {len(need_desc)}")

if need_desc:
    prompts = [
        f"Napisz zwięzły opis produktu (2-3 zdania, max 150 znaków) dla hurtowni budowlanej Media Bud w Lublinie. Produkt: {p['name']}. Kategoria: {p.get('catName', '')}. Język: profesjonalny polski, bez przymiotników marketingowych. Podaj tylko sam opis, bez wstępu."
        for p in need_desc
    ]
    
    short_descs = await batch_llm(prompts, system="Jesteś ekspertem od produktów budowlanych. Piszesz krótkie, faktyczne opisy dla hurtowni.")
    print(f"Wygenerowano {sum(1 for d in short_descs if d)} opisów")
    
    desc_updates = {}
    for p, desc in zip(need_desc, short_descs):
        if desc and len(desc) > 10:
            desc_updates[p['_id']] = desc.strip()
else:
    desc_updates = {}
    short_descs = []

print(f"Opis: {desc_updates.get(list(desc_updates.keys())[0], 'brak')[:100] if desc_updates else 'brak'}")

# ── Krok 4: Patchuj Sanity ────────────────────────────────────────────────────
print("\nAktualizacja Sanity...")
all_ids = set(list(unit_updates.keys()) + list(tag_updates.keys()) + list(desc_updates.keys()))
print(f"Produktów do aktualizacji: {len(all_ids)}")

success = 0
errors = 0
BATCH = 50

all_ids_list = list(all_ids)

async with aiohttp.ClientSession() as session:
    for i in range(0, len(all_ids_list), BATCH):
        batch = all_ids_list[i:i+BATCH]
        tasks = []
        for pid in batch:
            patch = {}
            if pid in unit_updates:
                patch['unit'] = unit_updates[pid]
            if pid in tag_updates:
                patch['tags'] = tag_updates[pid]
            if pid in desc_updates:
                patch['shortDescription'] = desc_updates[pid]
            if patch:
                tasks.append(sanity_patch(pid, patch, session))
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        ok = sum(1 for r in results if r is True)
        err = sum(1 for r in results if r is not True)
        success += ok
        errors += err
        
        if (i // BATCH) % 5 == 0:
            print(f"  Batch {i//BATCH+1}: {success} OK, {errors} błędów")

print(f"\n✅ Gotowe! Zaktualizowano: {success}, Błędów: {errors}")
print(f"  - Units: {len(unit_updates)}")
print(f"  - Tags: {len(tag_updates)}")
print(f"  - ShortDescs: {len(desc_updates)}")

# Zapisz podsumowanie
summary = {
    "timestamp": time.time(),
    "total_products": len(products),
    "unit_updates": len(unit_updates),
    "tag_updates": len(tag_updates),
    "desc_updates": len(desc_updates),
    "success": success,
    "errors": errors,
    "sample_units": dict(list(unit_updates.items())[:5]),
    "sample_tags": dict(list(tag_updates.items())[:3]),
}
with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/enrich_summary.json', 'w') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Zapisano: enrich_summary.json")


Pobieranie produktów z Sanity...


Pobrano: 3000 produktów
Bez shortDescription: 2984
Bez unit: 2984

Ekstrakcja jednostek z nazw produktów...
Produkty z wyciągniętą jednostką: 795
Produkty z wygenerowanymi tagami: 414

Generowanie krótkich opisów (batch_llm)...
Do generowania opisów: 0
Opis: brak

Aktualizacja Sanity...
Produktów do aktualizacji: 1107


  Batch 1: 50 OK, 0 błędów


  Batch 6: 300 OK, 0 błędów


  Batch 11: 550 OK, 0 błędów


  Batch 16: 800 OK, 0 błędów


  Batch 21: 1050 OK, 0 błędów



✅ Gotowe! Zaktualizowano: 1107, Błędów: 0
  - Units: 795
  - Tags: 414
  - ShortDescs: 0
Zapisano: enrich_summary.json


In [23]:

import asyncio, aiohttp, json, sys, re, time
sys.path.insert(0, "/app/skills/common/scripts")
from batch_llm import batch_llm

SANITY_PROJECT = "nzcwegq7"
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
DATASET = "production"

async def sanity_query(query):
    url = f"https://{SANITY_PROJECT}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
    async with aiohttp.ClientSession() as s:
        async with s.get(url, params={"query": query}, headers={"Authorization": f"Bearer {SANITY_TOKEN}"}) as r:
            return (await r.json()).get("result", [])

async def sanity_patch_many(updates: list, session):
    """updates: list of {id, patch_dict}"""
    url = f"https://{SANITY_PROJECT}.api.sanity.io/v2021-10-21/data/mutate/{DATASET}"
    mutations = [{"patch": {"id": u["id"], "set": u["set"]}} for u in updates]
    async with session.post(url, json={"mutations": mutations},
                            headers={"Authorization": f"Bearer {SANITY_TOKEN}", "Content-Type": "application/json"}) as r:
        return r.status == 200

# Pobierz produkty bez shortDescription - pierwsze 800 (dla szybkości)
print("Pobieranie produktów bez opisów...")
q = '*[_type == "product" && !defined(shortDescription) && defined(name)][0...800]{_id, name, "catSlug": category->slug.current, "catName": category->name, "brandName": brand->name}'
products = await sanity_query(q)
print(f"Produktów bez opisu: {len(products)}")

if not products:
    print("Brak produktów do generowania")
else:
    # Generuj opisy batchem
    prompts = [
        f"Napisz krótki opis produktu budowlanego dla hurtowni (1-2 zdania, max 120 znaków). Produkt: {p.get('name','')}. Kategoria: {p.get('catName','')}. Marka: {p.get('brandName','')}. Podaj tylko sam opis, po polsku, bez cudzysłowów."
        for p in products
    ]
    
    print(f"Generowanie {len(prompts)} opisów przez batch_llm...")
    descs = await batch_llm(prompts, system="Jesteś ekspertem produktów budowlanych. Piszesz zwięzłe, faktyczne opisy po polsku.")
    
    ok_descs = [(p, d) for p, d in zip(products, descs) if d and len(d.strip()) > 10]
    print(f"Opisów wygenerowanych: {len(ok_descs)}")
    
    # Patch Sanity w batchach po 200
    success = 0
    BATCH = 200
    async with aiohttp.ClientSession() as session:
        for i in range(0, len(ok_descs), BATCH):
            batch = ok_descs[i:i+BATCH]
            updates = [{"id": p["_id"], "set": {"shortDescription": d.strip()}} for p, d in batch]
            ok = await sanity_patch_many(updates, session)
            if ok:
                success += len(batch)
            print(f"  Batch {i//BATCH+1}: {success} OK")
    
    print(f"\n✅ Zaktualizowano {success} opisów")
    
    # Pokaż przykłady
    print("\nPrzykładowe opisy:")
    for p, d in ok_descs[:5]:
        print(f"  [{p.get('name','')[:40]}] → {d[:80]}")


Pobieranie produktów bez opisów...


Produktów bez opisu: 800
Generowanie 800 opisów przez batch_llm...


Batch complete: 800/800 succeeded
Opisów wygenerowanych: 800


  Batch 1: 200 OK


  Batch 2: 400 OK


  Batch 3: 600 OK


  Batch 4: 800 OK

✅ Zaktualizowano 800 opisów

Przykładowe opisy:
  [Lepik Do Kon.10Kg Jarlep K Roztwór Do] → Lepik do konserwacji 10 kg Jarlep K – roztwór do hydroizolacji.
  [Gładź 5Kg C-35] → Cekol Gładź 5kg C-35 - idealna do wykańczania ścian i sufitów. Łatwa w aplikacji
  [Klej Do Styr.25Kg Grip S Atlas Do Styr. ] → Klej Atlas Gris 25kg do styropianu białego i grafitowego. Mocny i wydajny.
  [Styr.100 Aqua Thermo Standard Proste Eps] → Styropian Arsanit Aqua Thermo Standard L036. Wysoka izolacyjność cieplna i akust
  [Wylewka Samopoz.4310 25Kg Weber] → Weber Wylewka samopoziomująca 4310 25kg – szybkie wyrównanie podłóg.


In [27]:

import aiohttp, json

SANITY_PROJECT = "nzcwegq7"
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
DATASET = "production"
URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
HDRS = {"Authorization": f"Bearer {SANITY_TOKEN}"}

async def q(query, params=None):
    p = {"query": query}
    if params:
        for k, v in params.items():
            p[f"${k}"] = json.dumps(v)
    async with aiohttp.ClientSession() as s:
        async with s.get(URL, params=p, headers=HDRS) as r:
            d = await r.json()
            if "error" in d:
                return f"ERROR: {d['error']}"
            return d.get("result")

# Test 1: czy kategoria chemia-budowlana istnieje i ma dzieci?
cat = await q('*[_type=="category" && slug.current=="chemia-budowlana"][0]{_id, "slug":slug.current, name, "children":*[_type=="category" && parent._ref==^._id]{"slug":slug.current, name, "children2":*[_type=="category" && parent._ref==^._id]{"slug":slug.current}}}')
print("Kategoria chemia-budowlana:")
print(f"  _id: {cat.get('_id') if cat else 'NIE ZNALEZIONO'}")
if cat:
    print(f"  dzieci L2: {[c['slug'] for c in cat.get('children',[])[:5]]}")
    total_l3 = sum(len(c.get('children2',[])) for c in cat.get('children',[]))
    print(f"  suma L3 podkategorii: {total_l3}")

print()

# Test 2: Ile produktów znajdzie stara vs nowa query
slugs_test = ["chemia-budowlana"]

# Stara query (bezpieczna)
old_q = '*[_type == "product" && category->slug.current in $slugs][0...5]{_id, name, "catSlug": category->slug.current}'
old_result = await q(old_q, {"slugs": slugs_test})
print(f"Stara query (tylko category->slug): {len(old_result) if isinstance(old_result, list) else old_result}")

# Nowa query z parent chain
new_q = '''*[_type == "product" && (
  category->slug.current in $slugs ||
  category->parent->slug.current in $slugs ||
  category->parent->parent->slug.current in $slugs ||
  category->parent->parent->parent->slug.current in $slugs
)][0...5]{_id, name, "catSlug": category->slug.current}'''
new_result = await q(new_q, {"slugs": slugs_test})
print(f"Nowa query (parent chain): {len(new_result) if isinstance(new_result, list) else new_result}")


Kategoria chemia-budowlana:
  _id: cat-chemia
  dzieci L2: ['gipsy-gladzie', 'grunty', 'hydroizolacje', 'kleje', 'tynki']
  suma L3 podkategorii: 10



Stara query (tylko category->slug): 0
Nowa query (parent chain): 5


In [31]:

import aiohttp, json

SANITY_PROJECT = "nzcwegq7"
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
DATASET = "production"
URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
HDRS = {"Authorization": f"Bearer {SANITY_TOKEN}"}

async def q(query, params=None):
    p = {"query": query}
    if params:
        for k, v in params.items():
            p[f"${k}"] = json.dumps(v)
    async with aiohttp.ClientSession() as s:
        async with s.get(URL, params=p, headers=HDRS) as r:
            d = await r.json()
            if "error" in d:
                return f"ERROR: {d.get('error', {}).get('description', str(d))}"
            return d.get("result")

# CATEGORY_BY_SLUG_QUERY (dokładnie jak w queries.ts po zmianie)
cat_query = '''*[_type == "category" && slug.current == $slug][0] {
  _id, "slug": slug.current, name, icon, description, order,
  "parentSlug": parent->slug.current,
  "parentName": parent->name,
  "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
    _id, "slug": slug.current, name, order,
    "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
      _id, "slug": slug.current, name, order,
      "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
        _id, "slug": slug.current, name, order
      }
    }
  }
}'''

cat_result = await q(cat_query, {"slug": "chemia-budowlana"})
if isinstance(cat_result, str):
    print(f"BŁĄD QUERY: {cat_result}")
elif cat_result is None:
    print("Kategoria NIE ZNALEZIONA")
else:
    print(f"Kategoria: {cat_result.get('name')} ({cat_result.get('slug')})")
    children = cat_result.get("children", [])
    print(f"  Dzieci L2: {len(children)}")
    for c in children[:5]:
        l3 = c.get("children", [])
        print(f"    L2: {c['slug']} → {len(l3)} L3 podkategorii")

    # Symuluj collectAllSlugs
    def collect_all_slugs(cat):
        slugs = [cat["slug"]]
        for child in cat.get("children", []):
            slugs.extend(collect_all_slugs(child))
        return slugs
    
    all_slugs = collect_all_slugs(cat_result)
    print(f"\n  allSubSlugs ({len(all_slugs)} slugów): {all_slugs[:10]}...")

    # Teraz przetestuj produkty query z tymi slugami
    prod_query = '''*[_type == "product" && (
      category->slug.current in $slugs ||
      category->parent->slug.current in $slugs ||
      category->parent->parent->slug.current in $slugs ||
      category->parent->parent->parent->slug.current in $slugs
    )] | order(name asc) {_id, name, "catSlug": category->slug.current}'''
    
    products = await q(prod_query, {"slugs": all_slugs})
    if isinstance(products, str):
        print(f"\nBŁĄD produkty query: {products}")
    else:
        print(f"\n  Produkty znalezione: {len(products) if products else 0}")
        if products:
            for p in products[:5]:
                print(f"    {p['name'][:50]} [{p.get('catSlug')}]")


Kategoria: Chemia budowlana (chemia-budowlana)
  Dzieci L2: 44
    L2: bloczek-komorkowy → 0 L3 podkategorii
    L2: cement → 0 L3 podkategorii
    L2: ceramika-budowlana → 0 L3 podkategorii
    L2: folie-paroprzepuszczalne → 0 L3 podkategorii
    L2: folie-w-plynie → 0 L3 podkategorii

  allSubSlugs (55 slugów): ['chemia-budowlana', 'bloczek-komorkowy', 'cement', 'ceramika-budowlana', 'folie-paroprzepuszczalne', 'folie-w-plynie', 'fugi', 'gladzie-proszek', 'grunty-pod-tynki', 'grunty-tynki']...



  Produkty znalezione: 1678
    Aceton 0,5L T [rozpuszczalniki]
    Aceton 0,5L Techniczny [rozpuszczalniki]
    Aceton 5,0L T [rozpuszczalniki]
    Aceton 5,0L Techniczny [rozpuszczalniki]
    Aceton Plastik 0,5L [rozpuszczalniki]


In [35]:

import aiohttp, json, time

SANITY_PROJECT = "nzcwegq7"
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
DATASET = "production"
URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
HDRS = {"Authorization": f"Bearer {SANITY_TOKEN}"}

async def q_timed(label, query, params=None):
    p = {"query": query}
    if params:
        for k, v in params.items():
            p[f"${k}"] = json.dumps(v)
    t0 = time.time()
    async with aiohttp.ClientSession() as s:
        async with s.get(URL, params=p, headers=HDRS) as r:
            elapsed = time.time() - t0
            data = await r.json()
            status = r.status
            err = data.get("error")
            result = data.get("result")
            n = len(result) if isinstance(result, list) else ("obj" if result else "null")
            if err:
                print(f"❌ {label}: status={status}, {elapsed:.2f}s, ERROR: {err.get('description','?')[:120]}")
            else:
                print(f"✅ {label}: {n} wyników, {elapsed:.2f}s")
            return result, err

# Test 1: Nowy ALL_CATEGORIES_QUERY (4 poziomy zagnieżdżeń)
all_cats_q = '''*[_type == "category" && !defined(parent)] | order(order asc, name asc) {
  _id, "slug": slug.current, name, icon, description, order,
  "parentSlug": parent->slug.current,
  "parentName": parent->name,
  "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
    _id, "slug": slug.current, name, order,
    "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
      _id, "slug": slug.current, name, order,
      "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
        _id, "slug": slug.current, name, order
      }
    }
  }
}'''

r1, e1 = await q_timed("ALL_CATEGORIES_QUERY (4 poziomy)", all_cats_q)

# Test 2: Uproszczona wersja (2 poziomy) — szybsza + bezpieczna
all_cats_2lvl = '''*[_type == "category" && !defined(parent)] | order(order asc, name asc) {
  _id, "slug": slug.current, name, icon, description, order,
  "parentSlug": parent->slug.current,
  "parentName": parent->name,
  "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
    _id, "slug": slug.current, name, order,
    "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
      _id, "slug": slug.current, name, order
    }
  }
}'''
r2, e2 = await q_timed("ALL_CATEGORIES_QUERY (2 poziomy)", all_cats_2lvl)

# Test 3: CATEGORY_BY_SLUG z 4 poziomami
cat_4lvl = '''*[_type == "category" && slug.current == $slug][0] {
  _id, "slug": slug.current, name, icon, description, order,
  "parentSlug": parent->slug.current, "parentName": parent->name,
  "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
    _id, "slug": slug.current, name, order,
    "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
      _id, "slug": slug.current, name, order,
      "children": *[_type=="category" && parent._ref == ^._id] | order(order asc, name asc) {
        _id, "slug": slug.current, name, order
      }
    }
  }
}'''
r3, e3 = await q_timed("CATEGORY_BY_SLUG chemia (4 poziomy)", cat_4lvl, {"slug": "chemia-budowlana"})

print()
if r1 and not e1:
    total_children = sum(len(c.get("children", [])) for c in r1)
    total_l3 = sum(len(c2.get("children", [])) for c in r1 for c2 in c.get("children", []))
    print(f"ALL_CATS: {len(r1)} top-level, {total_children} L2, {total_l3} L3")


✅ ALL_CATEGORIES_QUERY (4 poziomy): 9 wyników, 0.33s


✅ ALL_CATEGORIES_QUERY (2 poziomy): 9 wyników, 0.31s
✅ CATEGORY_BY_SLUG chemia (4 poziomy): obj wyników, 0.14s

ALL_CATS: 9 top-level, 130 L2, 10 L3


In [39]:

import aiohttp, json, urllib.parse

SANITY_PROJECT = "nzcwegq7"
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
DATASET = "production"

# Exact slugs z chemia-budowlana (55 slugów jak allSubSlugs)
all_slugs = ["chemia-budowlana","bloczek-komorkowy","cement","ceramika-budowlana",
             "folie-paroprzepuszczalne","folie-w-plynie","fugi","gladzie-proszek",
             "grunty-pod-tynki","grunty-tynki","hydroizolacje","izolacje-termiczne-chemiczne",
             "kleje","kleje-do-glazury","kleje-do-styropianu","kleje-montazowe",
             "masy-szpachlowe","masy-wyrownujace","materialy-wykonczen","piany-pistoletowe",
             "posadzki-chemiczne","progi-dylatacje","rozpuszczalniki","silikony-sanitarne",
             "silikony-uszczelniajace","styropian-chemiczny","sufit-klejony","szpachlowki",
             "tynki","tynki-gipsowe","tynki-maszynowe","tynki-mineralne","tynki-mozaikowe",
             "tynki-silikatowe","tynki-silikonowe","tynki-akrylowe","uszczelniacze",
             "wylewki-anhydrytowe","wylewki-samopoziomujace","zaprawy-cementowe",
             "zaprawy-murarskie","zaprawy-naprawcze","zaprawy-szlachetne","zbroje-siatki"]

# Test query
PRODUCT_CARD_FIELDS = """{
  _id, "id": _id,
  "slug": slug.current,
  name, sku, unit, featured, inStock,
  shortDescription, tags,
  "categorySlug": category->slug.current,
  "categoryName": category->name,
  "brand": brand->name,
  "images": images[].asset->url
}"""

query = f"""*[_type == "product" && (
  category->slug.current in $slugs ||
  category->parent->slug.current in $slugs ||
  category->parent->parent->slug.current in $slugs ||
  category->parent->parent->parent->slug.current in $slugs
)] | order(name asc) {PRODUCT_CARD_FIELDS}"""

# Sposób Sanity JS client: $slugs jako osobny parametr
url = f"https://{SANITY_PROJECT}.api.sanity.io/v2024-01-01/data/query/{DATASET}"
params = {
    "query": query,
    "$slugs": json.dumps(all_slugs)
}

import time
t0 = time.time()
async with aiohttp.ClientSession() as s:
    async with s.get(url, params=params,
                     headers={"Authorization": f"Bearer {SANITY_TOKEN}"}) as r:
        elapsed = time.time() - t0
        data = await r.json()
        status = r.status
        result = data.get("result", [])
        err = data.get("error")
        
        if err:
            print(f"❌ BŁĄD ({status}, {elapsed:.2f}s): {err}")
        elif result is None:
            print(f"⚠️ result=null ({elapsed:.2f}s), status={status}")
            print(f"Raw: {str(data)[:300]}")
        else:
            print(f"✅ {len(result)} produktów ({elapsed:.2f}s), status={status}")
            if result:
                print(f"Przykład: {result[0].get('name')} [{result[0].get('categorySlug')}]")


✅ 1679 produktów (2.67s), status=200
Przykład: Aceton 0,5L T [rozpuszczalniki]


In [43]:

import aiohttp, json

SANITY_PROJECT = "nzcwegq7"
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
DATASET = "production"
URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2024-01-01/data/query/{DATASET}"
HDRS = {"Authorization": f"Bearer {SANITY_TOKEN}"}

# Nowa prosta query z limitem 600
all_slugs = ["chemia-budowlana","bloczek-komorkowy","cement","ceramika-budowlana",
             "folie-paroprzepuszczalne","folie-w-plynie","fugi","gladzie-proszek",
             "grunty-pod-tynki","grunty-tynki","hydroizolacje","izolacje-termiczne-chemiczne",
             "kleje","kleje-do-glazury","kleje-do-styropianu","kleje-montazowe",
             "masy-szpachlowe","masy-wyrownujace","materialy-wykonczen","piany-pistoletowe",
             "posadzki-chemiczne","progi-dylatacje","rozpuszczalniki","silikony-sanitarne",
             "silikony-uszczelniajace","styropian-chemiczny","sufit-klejony","szpachlowki",
             "tynki","tynki-gipsowe","tynki-maszynowe","tynki-mineralne","tynki-mozaikowe",
             "tynki-silikatowe","tynki-silikonowe","tynki-akrylowe","uszczelniacze",
             "wylewki-anhydrytowe","wylewki-samopoziomujace","zaprawy-cementowe",
             "zaprawy-murarskie","zaprawy-naprawcze","zaprawy-szlachetne","zbroje-siatki"]

import time
q = '*[_type == "product" && category->slug.current in $slugs] | order(name asc) [0...600] {_id, name, "catSlug": category->slug.current, unit, tags}'
t0 = time.time()
async with aiohttp.ClientSession() as s:
    async with s.get(URL, params={"query": q, "$slugs": json.dumps(all_slugs)}, headers=HDRS) as r:
        data = await r.json()
        elapsed = time.time() - t0
        result = data.get("result", [])
        err = data.get("error")
        if err:
            print(f"BŁĄD: {err}")
        else:
            print(f"✅ {len(result)} produktów w {elapsed:.2f}s")
            # Sprawdź tagi i jednostki
            with_unit = sum(1 for p in result if p.get('unit'))
            with_tags = sum(1 for p in result if p.get('tags'))
            print(f"  Z unit: {with_unit}, z tags: {with_tags}")
            # Podgląd
            for p in result[:3]:
                print(f"  {p['name'][:50]} | unit={p.get('unit')} | tags={p.get('tags',[])} ")


✅ 600 produktów w 1.82s
  Z unit: 87, z tags: 47
  Aceton 0,5L T | unit=None | tags=None 
  Aceton 0,5L Techniczny | unit=None | tags=None 
  Aceton 5,0L T | unit=None | tags=None 
